In [4]:
#!pip install -e ..

In [1]:
from stable_baselines3 import PPO, TD3
from skopt import gp_minimize, gbrt_minimize 
from sb3_contrib import TQC, RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from rl4greencrab import evaluate_agent, multiConstAction, simulator
import pandas as pd
import numpy as np
from rl4greencrab import plot_agent
import ray
from skopt import gp_minimize, gbrt_minimize 
from skopt.plots import plot_convergence, plot_objective
from rl4greencrab import TwoActNormalized, twoActEnv

In [2]:
def evaluateConstAct(x):
    config = {
        'random_start':True,
        'var_penalty_const': 0,
        'observation_type': 'count-biomass-time',
        'control_randomness': True
    }
    env = twoActEnv(config)
    agent = multiConstAction(env=env, action=np.array(x))
    m_reward = evaluate_agent(agent=agent, ray_remote=True).evaluate(n_eval_episodes=200)
    
    return - m_reward

In [ ]:
%%time
max_action = 3000
ray.init(
    num_cpus=30,                 
    include_dashboard=False,   
    ignore_reinit_error=True
)
res = gp_minimize(evaluateConstAct, 2*[(0.0, max_action)], n_calls = 150, verbose=True)
res.x
ray.shutdown()

2026-06-08 22:02:04,849	INFO worker.py:2012 -- Started a local Ray instance.
/home/jovyan/miniconda/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/home/jovyan/miniconda/lib/python3.13/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Iteration No: 1 started. Evaluating function at random point.
False
{'object_store_memory': 2028184780.0, 'node:__internal_head__': 1.0, 'CPU': 30.0, 'node:10.42.0.54': 1.0, 'memory': 4732431156.0}


In [14]:
res.x

[1442.0569452909665, 366.1851707583983]

In [15]:
ray.init(
    num_cpus=30,                 
    include_dashboard=False,   
    ignore_reinit_error=True
)
rew = evaluateConstAct(
    [1442.0569452909665, 366.1851707583983]
    # [0.0, 787.3281177947351]
)
print(rew)
ray.shutdown()

2026-04-01 07:56:43,879	INFO worker.py:2012 -- Started a local Ray instance.
/opt/conda/lib/python3.12/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


False
{'CPU': 30.0, 'node:10.42.0.159': 1.0, 'object_store_memory': 9222386073.0, 'node:__internal_head__': 1.0, 'memory': 21518900839.0}
8.3487946579047


In [ ]:
constant_agent = multiConstAction(env=env, action=np.array(x))